<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

# Questão 1

Utilize o dataset Iris disponível no scikit-learn.
Divida os dados em treino e teste utilizando divisão estratificada.

**Solução**:

In [ ]:
# Utilize o dataset Iris disponível no scikit-learn.
# Divida os dados em treino e teste utilizando divisão estratificada.

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from collections import Counter

iris = load_iris()

X = iris.data      
y = iris.target    

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Original:", Counter(y))
print("Treino:", Counter(y_train))
print("Teste:", Counter(y_test))

# Questão 2

Treine um modelo utilizando `DecisionTreeClassifier`.

Depois calcule:

- acurácia no treino
- acurácia no teste

**Solução**:

In [ ]:
# Treine um modelo utilizando DecisionTreeClassifier.
# Depois calcule:
#   acurácia no treino
#   acurácia no teste

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_train = dt.predict(X_train)
y_pred_test  = dt.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test,  y_pred_test)

print(f"Acurácia no treino: {acc_train:.4f}")
print(f"Acurácia no teste:  {acc_test:.4f}")

# Questão 3

Utilize `plot_tree()` para visualizar a árvore treinada.

Responda:

1. Qual atributo aparece na raiz?
2. Qual é a profundidade da árvore?

**Solução**:

In [4]:
# 1. Qual atributo aparece na raiz?
# 2. Qual é a profundidade da árvore?

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree — Iris Dataset")
plt.tight_layout()
plt.show()

print(f"Profundidade da árvore: {dt.get_depth()}")
print(f"Atributo na raiz: {iris.feature_names[dt.tree_.feature[0]]}")

ModuleNotFoundError: No module named 'matplotlib'

**Respostas:**

1. **Atributo na raiz:** petal length(cm) aparece na raiz. Com um único corte em ≤ 2.45 cm já separa setosa das demais com 100% de pureza.

2. **Profundidade:** 5. Uma árvore sem restrição de max_depth cresce até que todos os nós sejam puros, explicando o overfitting.

# Questão 4

Treine dez árvores com:

- max_depth = 1
- max_depth = 2
- max_depth = 3
...
- max_depth = 9
- max_depth = None

Registre em uma tabela para cada árvore:

- acurácia no treino
- acurácia no teste
- profundidade da árvore
- número de folhas

**Solução**:

In [ ]:
# Treine dez árvores com max_depth de 1 a 9 e None.
# Registre acurácia no treino, no teste, profundidade e número de folhas.

import pandas as pd

depths = list(range(1, 10)) + [None]
records = []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)

    records.append({
        "max_depth":    depth,
        "profundidade": clf.get_depth(),
        "n_folhas":     clf.get_n_leaves(),
        "acc_treino":   accuracy_score(y_train, clf.predict(X_train)),
        "acc_teste":    accuracy_score(y_test,  clf.predict(X_test)),
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

**Em qual profundidade começa o overfitting?**

A partir de max_depth 4, o treino continua subindo enquanto o teste estagna ou cai, característica de overfitting. O ponto de melhor equilíbrio é max depth 3.

**Por que a árvore consegue 100% no treino quando max_depth=None?**

Sem restrição de profundidade, a árvore cresce até que todas as folhas sejam puras, cada folha contém amostras de apenas uma classe. É a definição de overfitting.

# Questão 5

Treine dois modelos:

- criterion = "gini"
- criterion = "entropy"

Compare:

- profundidade da árvore
- acurácia

**Solução**:

In [ ]:
# Treine dois modelos com criterion='gini' e criterion='entropy'.
# Compare profundidade e acurácia.

for criterion in ["gini", "entropy"]:
    clf = DecisionTreeClassifier(criterion=criterion, random_state=42)
    clf.fit(X_train, y_train)

    acc_treino = accuracy_score(y_train, clf.predict(X_train))
    acc_teste  = accuracy_score(y_test,  clf.predict(X_test))

    print(f"criterion={criterion!r:10s} | profundidade={clf.get_depth()} | "
          f"acc_treino={acc_treino:.4f} | acc_teste={acc_teste:.4f}")

**Análise:**

No dataset Iris os dois critérios chegam ao mesmo resultado. Isso ocorre porque Gini e Entropy são matematicamente muito similares: ambos medem impureza de nó e tendem a escolher os mesmos atributos para divisão. A diferença aparece em datasets maiores e com muitas classes.

# Questão 6

Escolha um hiperparâmetro e investigue seu impacto.

Sugestões:

- max_depth
- min_samples_split
- min_samples_leaf
- criterion

Mostre resultados e interprete.
- melhor modelo encontrado
- acurácia
- parâmetros

**Solução**:

In [ ]:
# Escolha um hiperparâmetro e investigue seu impacto no desempenho da árvore.
# Hiperparâmetro investigado: min_samples_leaf (de 1 a 20)

import pandas as pd

records = []

for min_samples_leaf in range(1, 21):
    clf = DecisionTreeClassifier(min_samples_leaf=min_samples_leaf, random_state=42)
    clf.fit(X_train, y_train)

    records.append({
        "min_samples_leaf": min_samples_leaf,
        "profundidade":     clf.get_depth(),
        "n_folhas":         clf.get_n_leaves(),
        "acc_treino":       accuracy_score(y_train, clf.predict(X_train)),
        "acc_teste":        accuracy_score(y_test,  clf.predict(X_test)),
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

best = df.loc[df["acc_teste"].idxmax()]
print(f"\nMelhor modelo:")
print(f"  min_samples_leaf : {int(best['min_samples_leaf'])}")
print(f"  profundidade     : {int(best['profundidade'])}")
print(f"  n_folhas         : {int(best['n_folhas'])}")
print(f"  acc_treino       : {best['acc_treino']:.4f}")
print(f"  acc_teste        : {best['acc_teste']:.4f}")

min_samples_leaf controla o tamanho mínimo de cada folha.

- **Valores baixos (1–2):** árvore mais profunda, overfitting presente.
- **Valores médios (3–4):** treino cai levemente mas teste sobe, zona ideal de generalização.
- **Valores altos (≥11):** underfitting, folhas muito grandes forçando a arvore a simplificar.

O melhor modelo encontrado usa min_samples_leaf=3, com profundidade 3, apenas 5 folhas, sem overfitting e maior precisão no teste.